### Virtual-ISP Learning Notebook

This notebook guides you through exploring, preprocessing, modeling, and evaluating data in the Virtual-ISP project. You can experiment with RAW image processing and basic machine learning concepts step by step.

### Import Required Libraries
Import essential libraries for image processing and machine learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rawpy

# 1. Decode & Extract Data

In [ ]:
def decode_arw_image(file_path):
    with rawpy.imread(file_path) as raw:
        bayer = raw.raw_image_visible.copy()  # 2D Bayer mosaic (decoded RAW data)
        cfa = raw.raw_pattern.copy()  # CFA layout, [[0,1], [1,2]]
        black = np.array(raw.black_level_per_channel)
        white = raw.white_level
        color_desc = raw.color_desc

    return bayer, cfa, black, white, color_desc

# 2. Linearize
Each pixel is a sensor intensity fraction that we set between 0-1. If a value reads ~0.0 that pixel is at/under black level (no useful light signal). 
If the value reads ~1, that pixel is at/over sensor white level (saturated). 
Values in the middle like .25 or .6 are proportional light measurements. 

We calculate linear by accepting the bayer, black & white level from the decoded sensor values. 
Then we calculate by subtracting the bayer by the black scalar... dividing the result by the difference of white level and black scalar. 

We use np.clip to limit the values between 0 and 1. 

In [ ]:
import numpy as np
import rawpy
import matplotlib.pyplot as plt

In [ ]:
def linearize_bayer(bayer, black_level, white_level):
    black_scalar = float(black_level[0])
    linear = (bayer.astype(np.float32) - black_scalar) / \
        (white_level - black_scalar)
    return np.clip(linear, 0.0, 1.0) # limit linear values between 0 and 1



bayer, cfa, black, white, color_desc = decode_arw_image('.\imgs\AKG02229.ARW')
linear = linearize_bayer(bayer, black, white) # each pixel is a sensor intensity fraction

print(linear)

# 3. Use the CFA as a repeating “label map” over the full Bayer image:

raw_pattern is a 2x2 tile of channel IDs. That 2x2 tile repeats across the whole sensor and this pattern **never changes**. 

Each ID maps to a color via color_desc (often like RGBG, so two IDs can both be green).

Build boolean masks for R/G/B, then pull values from linear into channel planes.

In [ ]:
def build_rgb_masks(bayer, cfa, color_desc):
    
    h, w = bayer.shape


    full_ids = np.tile(cfa, (h // 2 + 1, w // 2 + 1))[:h, :w]

    red_mask = (full_ids == color_desc.index(b'R'))
    green_mask = (full_ids == color_desc.index(b'G'))
    blue_mask = (full_ids == color_desc.index(b'B'))
    return red_mask, green_mask, blue_mask

## Explanation 
What is index()? It's a built-in method on Python bytes objects (same as it is on lists/strings). It searches for a value and returns its position.

b'RGBG'.index(b'R') → 0.

b'RGBG'.index(b'B') → 2

In [ ]:
## Putting it all together

color_desc = b'RGBG'
             index: 0 1 2 3

color_desc.index(b'R') → 0
color_desc.index(b'G') → 1   (first G found)
color_desc.index(b'B') → 2

full_ids = [[0, 1],   →  [[R, G],
            [1, 2]]          [G, B]]

red_mask  = full_ids == 0  →  True only at R positions
green_mask = full_ids == 1  →  True at both G positions
blue_mask = full_ids == 2  →  True only at B positions


red_mask          green_mask        blue_mask
T F F F T F       F T T F F T       F F F T F F
F F F F F F       F T T F F T       F F F F F F
T F F F T F       F T T F F T       F F F T F F
...               ...               ...

 # Each True position means "this pixel belongs to this channel". The Trues never overlap — every pixel is owned by exactly one channel

# Demosaicing

We will need to perform interpolation. 

*Interpolation - estimating a value between known values.*

## Demosaic methods

* **Bilinear**: easiest, fast, good for learning. Works by estimating each missing pixel in two directions (horizontal & vertical)
* **Nearest-neighbor**: even simpler, usually lower quality  
* **Edge-aware / gradient-based (e.g. Malvar, AHD):** preserve detail better, reduce color artifacts
* **Modern/ML methods:** best quality in some cases, more complex

We will start with a bilinear demosaic for learning purposes: 